# Non-cubic CUDA/JAX dose viewer

This notebook views the fresh `(z, y, x) = (96, 112, 128)` ring calculation without modifying any historical reference file. Generate or refresh the inputs from the repository root with:

```bash
source scripts/activate-dosecuda.sh
python tests/render_noncubic_comparison.py
```

Select the project `.venv` as the notebook kernel, then use the axis buttons and slice slider below.

In [4]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk
from IPython.display import display

repository_root = Path.cwd()
if repository_root.name == 'tests':
    repository_root = repository_root.parent

output_dir = repository_root / 'test_phantom_output' / 'noncubic_indexing'
paths = {
    'CT': output_dir / 'noncubic_phantom_ct.nrrd',
    'CUDA': output_dir / 'noncubic_impt_dose_cuda.nrrd',
    'JAX': output_dir / 'noncubic_impt_dose_jax.nrrd',
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Generate the visualization inputs first. Missing: ' + ', '.join(missing))

images = {name: sitk.ReadImage(str(path)) for name, path in paths.items()}
volumes = {name: sitk.GetArrayFromImage(image).astype(np.float32, copy=False) for name, image in images.items()}
ct, cuda, jax = volumes['CT'], volumes['CUDA'], volumes['JAX']
difference = jax - cuda
origin_xyz = np.asarray(images['CT'].GetOrigin())
spacing_xyz = np.asarray(images['CT'].GetSpacing())

print(f'array shape (z, y, x): {ct.shape}')
print(f'physical origin (x, y, z): {tuple(origin_xyz)}')
print(f'spacing (x, y, z): {tuple(spacing_xyz)} mm')
print(f'CUDA peak: {cuda.max():.9g} at {np.unravel_index(cuda.argmax(), cuda.shape)}')
print(f'JAX peak:  {jax.max():.9g} at {np.unravel_index(jax.argmax(), jax.shape)}')
print(f'max |JAX-CUDA|: {np.abs(difference).max():.9g}')
print(f'correlation: {np.corrcoef(cuda.ravel(), jax.ravel())[0, 1]:.12g}')

array shape (z, y, x): (96, 112, 128)
physical origin (x, y, z): (-192.0, -168.0, -144.0)
spacing (x, y, z): (3.0, 3.0, 3.0) mm
CUDA peak: 0.854743779 at (79, 24, 60)
JAX peak:  0.854932129 at (79, 24, 60)
max |JAX-CUDA|: 0.00090277195
correlation: 0.99999999255


In [5]:
axis_names = {0: 'z (axial)', 1: 'y (coronal)', 2: 'x (sagittal)'}

def plane_geometry(axis):
    nz, ny, nx = ct.shape
    x0, y0, z0 = origin_xyz
    sx, sy, sz = spacing_xyz
    if axis == 0:
        return (x0, x0 + (nx - 1) * sx, y0, y0 + (ny - 1) * sy), 'x (mm)', 'y (mm)', z0, sz
    if axis == 1:
        return (x0, x0 + (nx - 1) * sx, z0, z0 + (nz - 1) * sz), 'x (mm)', 'z (mm)', y0, sy
    return (y0, y0 + (ny - 1) * sy, z0, z0 + (nz - 1) * sz), 'y (mm)', 'z (mm)', x0, sx

def show_slice(axis, index):
    extent, xlabel, ylabel, coordinate_origin, coordinate_spacing = plane_geometry(axis)
    coordinate = coordinate_origin + index * coordinate_spacing
    dose_max = max(float(cuda.max()), float(jax.max()))
    difference_limit = float(np.max(np.abs(difference)))
    arrays = (ct, cuda, jax, difference)
    titles = ('CT', 'CUDA dose', 'JAX dose', 'JAX - CUDA')
    cmaps = ('gray', 'turbo', 'turbo', 'RdBu_r')

    figure, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
    for column, (plot_axis, volume, title, cmap) in enumerate(zip(axes, arrays, titles, cmaps)):
        image_slice = np.take(volume, index, axis=axis)
        if column == 0:
            limits = {'vmin': -1000.0, 'vmax': 200.0}
        elif column in (1, 2):
            limits = {'vmin': 0.0, 'vmax': dose_max}
        else:
            limits = {'vmin': -difference_limit, 'vmax': difference_limit}
        rendered = plot_axis.imshow(image_slice, cmap=cmap, origin='lower', extent=extent, aspect='equal', **limits)
        plot_axis.set_title(title)
        plot_axis.set_xlabel(xlabel)
        plot_axis.set_ylabel(ylabel)
        figure.colorbar(rendered, ax=plot_axis, fraction=0.046, pad=0.04)

    figure.suptitle(f'{axis_names[axis]} index {index} = {coordinate:.1f} mm')
    plt.show()

In [6]:
axis_selector = widgets.ToggleButtons(
    options=[('z / axial', 0), ('y / coronal', 1), ('x / sagittal', 2)],
    value=0,
    description='Plane:',
)
slice_slider = widgets.IntSlider(
    value=ct.shape[0] // 2,
    min=0,
    max=ct.shape[0] - 1,
    step=1,
    description='Slice:',
    continuous_update=False,
)

def update_slider(change):
    axis = change['new']
    slice_slider.max = ct.shape[axis] - 1
    slice_slider.value = ct.shape[axis] // 2

axis_selector.observe(update_slider, names='value')
viewer = widgets.interactive_output(show_slice, {'axis': axis_selector, 'index': slice_slider})
display(widgets.VBox([axis_selector, slice_slider]), viewer)

Output()